# Carga dos dados brutos para o volume
Download dos conjuntos de dados da ANAC e salvamento em volume no schema staging

In [0]:
%sql
USE CATALOG mvp_engenharia_dados;
USE SCHEMA staging;

## Imports

In [0]:
from time import sleep
import pandas as pd
import requests
import os
from pathlib import Path
import random

## Arquivos VRA
Os arquivos se encontram em um diretório com uma pasta para cada ano e uma pasta para cada mês em: https://siros.anac.gov.br/siros/registros/diversos/vra/

Vamos criar uma logica para consultar o nome dos arquivos para o periodo de Janeiro de 2023 a Julho de 2026.

Os arquivos seguem uma estrutura padrão, como por exemplo https://siros.anac.gov.br/siros/registros/diversos/vra/2022/VRA_2022_01.csv

Basta criar as urls com o mesmo padrão e consulta-las.

In [0]:
start_date = '2023-01-01'
end_date = '2026-07-01'

date_range = pd.date_range(start=start_date, end=end_date, freq='MS')

base_url = "https://siros.anac.gov.br/siros/registros/diversos/vra" 

urls = [
    f"{base_url}/{d.year}/VRA_{d.strftime('%Y_%m')}.csv"
    for d in date_range
]

print(f"Arquivos a solicitar de {start_date} a {end_date}: {len(urls)} arquivos.")
print(f"Primeiro: {urls[0]}")
print(f"Último: {urls[-1]}")

Vamos salvar em uma pasta dedicada no volume, e também utilizar um sleep com intervalo aleatorio para evitar de realizar muitas solicitações ao servidor do SIROS

In [0]:
target_dir = Path("/Volumes/mvp_engenharia_dados/staging/pontualidade/vra")

os.makedirs(target_dir, exist_ok=True)

for url in urls:
    file_name = Path(url).name
    target_path = target_dir.joinpath(file_name)
    print(f"Baixando: {url} para: {target_path}")

    response = requests.get(url)
    response.raise_for_status()

    with open(target_path, 'wb') as f:
        f.write(response.content)
    print(f"Arquivo salvo em: {target_path}")
    sleep(random.uniform(1.0, 3.0))

    

## Arquivos de dimensões

Os arquivos de aeroportos e companhias sao mais simples pois são apenas um de cada, com link direto.

In [0]:
link_aeroportos = "https://sistemas.anac.gov.br/dadosabertos/Aerodromos/Aer%C3%B3dromos%20P%C3%BAblicos/Lista%20de%20aer%C3%B3dromos%20p%C3%BAblicos/AerodromosPublicos.csv"
link_empresas = "https://sistemas.anac.gov.br/dadosabertos/Operador%20A%C3%A9reo/Empresas%20Aereas%20Nacionais/pda_empresas_aereas_nacionais.json"

base_dir = Path("/Volumes/mvp_engenharia_dados/staging/pontualidade/")

response = requests.get(link_aeroportos)
response.raise_for_status()

with open(base_dir / Path(link_aeroportos).name, 'wb') as f:
    f.write(response.content)
    print(f"Arquivo salvo em: {base_dir / Path(link_aeroportos).name}")

response = requests.get(link_empresas)
response.raise_for_status()

with open(base_dir / Path(link_empresas).name, 'wb') as f:
    f.write(response.content)
    print(f"Arquivo salvo em: {base_dir / Path(link_empresas).name}")
